## Start Here

This is the modeling notebook. We are compaing the directional prediction of the following models:
    
- Coin Flip (Baseline) - Simple Model
- Majority Direction - Simple Model
- 5-Day Markov Chain
- Logistic Regression (train/test)
- Random Forest (train/test)

The models will be stored in the Predictions table in the database.

## Notes

- Every 'run' is done for one stock
- This notebook will work for the functionality of the Model Runs table and the Predictions Table for my Database

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression # For Logistic
from sklearn.ensemble import RandomForestClassifier # Random Forest
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Connect to Database

In [ ]:
# Function
def db_connection(server_name, database_name):
    import pandas as pd
    from sqlalchemy import create_engine

    #server_name = r"DESKTOP-EE25GV9\SQLEXPRESS" # <----- Change This
    #database_name = "stockPredictionApp" # <----- Change This

    # Do not change 'server_name' or 'database_name'
    connection_string = (
        f"mssql+pyodbc://@{server_name}/{database_name}"
        "?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )

    engine = create_engine(connection_string) # Connection to Database

    print("Connection successful!")
    
    # Test Query - Find the different tickers
    query = "SELECT * FROM Stocks"

    stocks_df = pd.read_sql(query, engine)

    print("\n",stocks_df)
    return(engine)


# ======================================
# ===== For Azure Database (Cloud) =====
def azure_connection():
    
    from sqlalchemy import create_engine
    from urllib.parse import quote_plus

    server = "serverName"
    database = "DLA_StockPrediction"
    username = "username"
    password = "password"

    params = quote_plus(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"UID={username};"
        f"PWD={password};"
    )

    engine = create_engine(
        f"mssql+pyodbc:///?odbc_connect={params}"
    )

    return(engine)




# Call
engine = db_connection(server_name = r"DESKTOP-EE25GV9\SQLEXPRESS", database_name = "stockPredictionApp")

Connection successful!

    StockID Ticker       CompanyName      Sector
0        1    BAC   Bank of America  Financials
1        2    TFC  Truist Financial  Financials
2        3    WFC       Wells Fargo  Financials


# Query Database for Prices and Features

- This chunk of code uses a SQL query from the June 2026 Week 1 Queries.
- The query takes all of the features and the closing price from Bank of America (StockID = 1) and puts them in one dataframe

In [3]:
def query_data(engine, stock_id):
    import pandas as pd
    query = "SELECT f.*, p.ClosePrice FROM Features f LEFT JOIN Prices p ON p.StockID = f.StockID AND p.PriceDate = f.FeatureDate WHERE f.StockID =" + str(stock_id)
    df = pd.read_sql(query, engine)
    df['y'] = np.where(df['ClosePrice'].diff().shift(-1) > 0, 1, 0)
    df = df.dropna()
    print("Query Successful! \n")
    return(df)

# Call
#df = query_data(engine, stock_id = 2)
#engine = azure_connection()
#df = query_data(engine, stock_id = 2)
#print(df)

# Calculate Model Results

We will first get the coin flip and majority class predictions.

Columns for Prediction Dataframe: Each output is 0/1

- direction - the direction of the stock from yesterday's closing to today's closing
- nextDayDirection - the future direction of the stock
- coinFlip - the coin flip direction
- majorityClass - majority class for 5 days
- 

## Coin Flip and Direction of Stock Prices

### Important!!!!

- This code chunk creates the predictions in the same dataframe as the features!

In [4]:
def dir_coinFlip_Model(df):
    # Trip Dataframe for Future Modeling
    #df = df.iloc[199:].reset_index(drop=True)
    
    # Direction of Stock Prices and Creation of the Dataframe
    df['direction'] = np.where(df['ClosePrice'].diff() > 0, 1, 0) # <--------------------------------- Direction
    #predictions = pd.DataFrame() # ---------- Make Data Frame for Predictions
    #predictions['direction'] = direction
    
    # Next Day Direction
    #predictions['nextDayDirection'] = predictions['direction'].shift(-1) # <-------- Next Day Direction

    # Coin Flip
    n = len(df['ClosePrice']) # ---------- Total Number of Entries
    df['coinFlip'] = np.random.randint(0,2, size = n) # <---------------------------------------- Coin Flip Results

    #predictions.tail()

    return(df)

# Call
#df = dir_coinFlip_Model(df)
#df.head()

## Majority Class

In [5]:
def majority_Model(df, win = 5, threshold = 0.5): # 5-Day Window
    
    df['majorityClass'] = np.where(df['direction'].rolling(window = win).mean() > threshold, 1, 0) # Majority Class Prediction
    return(df)

# Call
#df = majority_Model(df)
#df.tail()

## Logistic Regression

In [6]:
# Logistic Regression
def logistic_model(df, test_perc = 0.3):
    from sklearn.metrics import accuracy_score

    feature_cols = [
    "SMA_50",
    "SMA_200",
    "RollingVariance_50",
    "RSI",
    "BollingerUpper",
    "BollingerLower",
    "ATR",
    "VolumeChange"
    ]

    X = df[feature_cols]

    # ============================================
    # Pre diagnostic
    # ============================================
    print("Rows in df:", len(df))
    print("Columns:")
    print(df.columns.tolist())

    print("\nTarget counts:")
    print(df['y'].value_counts(dropna=False))
    # =============================================
    
    # Prepare Data
    #X = df.iloc[:, 3:11]
    y = df['y']
    split_idx = int(len(df) * (1 - test_perc))
    print("Split Index:", split_idx)
    
    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]

    y_train = y.iloc[:split_idx]
    y_test = y.iloc[split_idx:]

    # =================================================
    # Post diagnostic
    # ===============================================
    print("X_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_train shape:", y_train.shape)
    print("y_test shape:", y_test.shape)
    #=============================================
    
    # Scale and Train
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    
    
    # Fit Model
    model = LogisticRegression()
    model.fit(
        X_train_scaled,
        y_train
    )

    # Predictions
    logisticPred = model.predict(X_test_scaled)

    df['logisticPred'] = np.nan
    df.loc[X_test.index, 'logisticPred'] = logisticPred

    # Quick Evaluation
    print("Accuracy", accuracy_score(y_test, logisticPred))
    
    return(df)

# Call
#df = logistic_model(df)
#df.tail()

## Random Forest (Need Hyperparameters)

In [7]:
def random_forest_model(df, test_perc = 0.3):
    
    from sklearn.metrics import accuracy_score

   
    feature_cols = [
    "SMA_50",
    "SMA_200",
    "RollingVariance_50",
    "RSI",
    "BollingerUpper",
    "BollingerLower",
    "ATR",
    "VolumeChange"
    ]

    X = df[feature_cols]
    
    # Prepare Data
    #X = df.iloc[:, 3:11]
    y = df['y']
    split_idx = int(len(df) * (1 - test_perc))

    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]

    y_train = y.iloc[:split_idx]
    y_test = y.iloc[split_idx:]

    # Scale and Train
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Initialize RF Classifier
    rf_model = RandomForestClassifier(
        n_estimators = 100,
        max_depth = None,
        random_state = 42
    )
    
    # Train Model
    rf_model.fit(X_train, y_train)
    
    
    # Predictions
    rfPred = rf_model.predict(X_test_scaled)

    df['rfPred'] = np.nan
    df.loc[X_test.index, 'rfPred'] = rfPred

    # Quick Evaluation
    print("Accuracy", accuracy_score(y_test, rfPred))
    
    return(df)

# Call
#df = random_forest_model(df)
#df.tail()

# Create Prediction Table

- To do this, we will use a concept called 'unpivoting'. instead of looping the predictions to make the dataframe, we will use pandas

In [8]:

def make_prediction_table(df):
    # Chose Prediction Columns
    prediction_columns = [
        "coinFlip",
        "majorityClass",
        "logisticPred",
        "rfPred"
    ]

    # Transform
    predictions_long = df.melt(
        id_vars = ["StockID", "FeatureDate", "y"],
        value_vars = prediction_columns,
        var_name = "ModelName",
        value_name = "Prediction"
    )

    # Dataframe
    #predictions_long.tail()

    # Rename FeatureDate Column to PredDate
    predictions_long = predictions_long.rename(
    
    columns={"FeatureDate":"PredDate", "y":"ActualDirection"}
    
    )

    return (predictions_long)# <----------- Return This


#pred = make_prediction_table(df)
#pred.head()

In [9]:
#df.head()

# Insert Into Database

In [10]:
def insert_sql(df, engine, stock_id):
    import pandas as pd
    from sqlalchemy import create_engine
    import yfinance as yf
    
    if stock_id == 1:
        df.to_sql(
        "Predictions",
        engine,
        if_exists="replace",
        index=False
    )
    else:
        df.to_sql(
        "Predictions",
        engine,
        if_exists="append",
        index=False
    )
    

    print("Predictions inserted successfully!")
    
    # Sample Query
    query = "SELECT TOP 10* FROM Predictions WHERE StockID = " + str(stock_id)
    df = pd.read_sql(query, engine)

    print(df.head())

#insert_sql(bac_features, engine, stock_id = 1)

# Final function to perform All Duties in this notebook

In [11]:
# =====================================================
# ========== Master Model ==========
    
    
# Libraries

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression # For Logistic
from sklearn.ensemble import RandomForestClassifier # Random Forest
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Organizational Function Template:
def run_all_models(stock_id):
    print(f"\n===== STOCK {stock_id} =====")
    
    # ==================================
    # ======= Database Connection ======
    
    # Create Database Connection (engine)
    engine = db_connection(server_name = r"DESKTOP-EE25GV9\SQLEXPRESS", database_name = "stockPredictionApp")
    #engine = azure_connection()
    
    # Querry Features
    df = query_data(engine, stock_id = stock_id)
    print("Features Queried! \n")
    
    # ==================================
    # ========== Train Models ==========
    df = dir_coinFlip_Model(df) # ----- Coin Flip
    df = majority_Model(df) # ----- Majority Class
    df = logistic_model(df) # ----- Logistic Regression
    df = random_forest_model(df) # ----- Random Forest
    
    # ==========================================================
    # ========== create model run (prediction) record ==========
    pred = make_prediction_table(df)
    
    # =====================================
    # ========== Insert into SQL ==========
    insert_sql(pred, engine, stock_id = stock_id)
    print("Predictions Finished!")
    
#run_all_models(stock_id = 1)

# Master Function Call

In [13]:
# Add a New Ticker Here!!!!
run_all_models(stock_id = 3)


===== STOCK 3 =====
Connection successful!

    StockID Ticker       CompanyName      Sector
0        1    BAC   Bank of America  Financials
1        2    TFC  Truist Financial  Financials
2        3    WFC       Wells Fargo  Financials
Query Successful! 

Features Queried! 

Rows in df: 1437
Columns:
['StockID', 'FeatureDate', 'SMA_50', 'SMA_200', 'RollingVariance_50', 'RSI', 'BollingerUpper', 'BollingerLower', 'ATR', 'VolumeChange', 'ClosePrice', 'y', 'direction', 'coinFlip', 'majorityClass']

Target counts:
y
1    748
0    689
Name: count, dtype: int64
Split Index: 1005
X_train shape: (1005, 8)
X_test shape: (432, 8)
y_train shape: (1005,)
y_test shape: (432,)
Accuracy 0.4675925925925926


C:\Users\EJ's\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Accuracy 0.49074074074074076
Predictions inserted successfully!
   StockID   PredDate  ActualDirection ModelName  Prediction
0        3 2020-10-15                0  coinFlip         1.0
1        3 2020-10-16                0  coinFlip         0.0
2        3 2020-10-19                1  coinFlip         0.0
3        3 2020-10-20                0  coinFlip         1.0
4        3 2020-10-21                1  coinFlip         1.0
Predictions Finished!
